In [1]:
from pathlib import Path
import xdem
from xdem.terrain import slope
import matplotlib.pyplot as plt
import numpy as np
import rasterio


In [22]:
dem_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/")
dem_paths = list(dem_folder.glob("*.tif"))
output_folder = dem_folder / "terrain_attributes"
output_folder.mkdir(exist_ok=True)


In [23]:
print(dem_paths)

[PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/alos_dem_utm32635.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/aster_dem_utm32635.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/copernicus_dеm_utm32635.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/fab_dem_utm32635.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/nasa_dem_utm32635.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/srtm_dem_utm32635.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/tan_dem_utm32635.tif')]


In [25]:

def plot_attribute(attribute, cmap, label=None, vlim=None):
    if vlim is not None:
        if isinstance(vlim, (int, float)):
            vlims = {"vmin": -vlim, "vmax": vlim}
        elif isinstance(vlim, (list, tuple)) and len(vlim) == 2:
            vlims = {"vmin": vlim[0], "vmax": vlim[1]}
        else:
            vlims = {}
    else:
        vlims = {}

    attribute.plot(cmap=cmap, cbar_title=label, **vlims)
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.show()


In [26]:
def save_raster(array, reference_path, output_path, nodata_value=None):
    """Зберігає масив як GeoTIFF, зберігаючи метадані з оригінального DEM."""
    with rasterio.open(reference_path) as src:
        profile = src.profile
    profile.update(dtype="float32", count=1, compress="lzw")
    if nodata_value is not None:
        profile.update(nodata=nodata_value)
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(array.astype("float32"), 1)


In [27]:
for path in dem_paths:
    name = path.stem
    dem = xdem.DEM(path)

    print(f"\n📂 {name}")

    slope_horn = dem.slope(method="Horn")
    save_raster(slope_horn.data, path, output_folder / f"{name}_slope.tif")

    aspect = dem.aspect(method="Horn")
    save_raster(aspect.data, path, output_folder / f"{name}_aspect.tif")

    curvature = dem.curvature()
    save_raster(curvature.data, path, output_folder / f"{name}_curvature.tif")

    tpi = dem.topographic_position_index()
    save_raster(tpi.data, path, output_folder / f"{name}_tpi.tif")

    tri = dem.terrain_ruggedness_index()
    save_raster(tri.data, path, output_folder / f"{name}_tri.tif")

    roughness = dem.roughness()
    save_raster(roughness.data, path, output_folder / f"{name}_roughness.tif")


    print(f"✅ Збережено всі атрибути для {name}")



📂 alos_dem_utm32635
✅ Збережено всі атрибути для alos_dem_utm32635

📂 aster_dem_utm32635
✅ Збережено всі атрибути для aster_dem_utm32635

📂 copernicus_dеm_utm32635
✅ Збережено всі атрибути для copernicus_dеm_utm32635

📂 fab_dem_utm32635
✅ Збережено всі атрибути для fab_dem_utm32635

📂 nasa_dem_utm32635
✅ Збережено всі атрибути для nasa_dem_utm32635

📂 srtm_dem_utm32635
✅ Збережено всі атрибути для srtm_dem_utm32635

📂 tan_dem_utm32635
✅ Збережено всі атрибути для tan_dem_utm32635
